# LAB1 – Bài 2: Yêu cầu lập trình
**Sinh viên:** Đinh Văn Bình – **MSSV:** 6351071007

## Câu 6: Tiền xử lý các thuộc tính thiếu dữ liệu và xóa bỏ các thuộc tính không cần thiết

In [1]:
print('Họ tên: Đinh Văn Bình - MSSV: 6351071007')

Họ tên: Đinh Văn Bình - MSSV: 6351071007


### Bước 1. Import thư viện và đọc dữ liệu

In [2]:
import re
import numpy as np
import pandas as pd

with open('titanic.csv') as f:
    df = pd.read_csv(f, dtype={'age': np.float64, 'sibsp': np.int64, 'parch': np.int64})

print('Kích thước dữ liệu ban đầu (dòng, cột):', df.shape)

Kích thước dữ liệu ban đầu (dòng, cột): (1309, 14)


### Bước 2. Xem số giá trị thiếu của từng thuộc tính trước khi xử lý

In [3]:
df.isnull().sum()

pclass          0
survived        0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

### Bước 3. Tách thông tin boong tàu (deck) từ thuộc tính cabin rồi xóa cabin
Các thuộc tính `body`, `cabin`, `boat`, `home.dest` thiếu rất nhiều giá trị và ít ảnh hưởng đến khả năng sống sót nên có thể xóa. Tuy nhiên, mỗi giá trị của `cabin` (ví dụ `C123`) có ký tự đầu `C` là boong tàu hành khách đang ở → dữ liệu này có thể hữu ích nên tách ra thành thuộc tính `deck` trước khi xóa.
- Cabin bị thiếu được gán `U0` (Unknown) → `deck` = 8.
- Boong không có trong bảng ánh xạ (ví dụ `T`) → `deck` = 0.

In [4]:
deck = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "U": 8}

for dataset in [df]:
    dataset['cabin'] = dataset['cabin'].fillna("U0")
    dataset['deck'] = dataset['cabin'].map(lambda x: re.compile("([a-zA-Z]+)").search(x).group())
    dataset['deck'] = dataset['deck'].map(deck)
    dataset['deck'] = dataset['deck'].fillna(0)
    dataset['deck'] = dataset['deck'].astype(int)
# we can now drop the cabin feature
df = df.drop(['cabin'], axis=1)
df['deck'].value_counts().sort_index()

deck
0       1
1      22
2      65
3      94
4      46
5      41
6      21
7       5
8    1014
Name: count, dtype: int64

### Bước 4. Xem xét thuộc tính ticket

In [5]:
df['ticket'].describe()

count         1309
unique         929
top       CA. 2343
freq            11
Name: ticket, dtype: object

Thuộc tính `ticket` có đến 929 giá trị khác nhau trên 1309 dòng → gần như mỗi người một mã vé, không giúp ích cho quá trình khai thác mà còn có thể gây sai số cho thuật toán → xóa đi.

### Bước 5. Xóa các thuộc tính không cần thiết
- `ticket`: quá nhiều giá trị khác nhau.
- `boat`, `body`: thiếu nhiều dữ liệu và là thông tin có **sau** khi biết kết quả sống sót (số xuồng cứu sinh, số hiệu thi thể) → giữ lại sẽ làm sai lệch mô hình.
- `home.dest`: thiếu 43% và không liên quan đến khả năng sống sót.

In [6]:
df = df.drop(['ticket'], axis=1)
df = df.drop(['boat'], axis=1)
df = df.drop(['body'], axis=1)
df = df.drop(['home.dest'], axis=1)
print('Các thuộc tính còn lại:', list(df.columns))

Các thuộc tính còn lại: ['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'deck']


### Bước 6. Điền giá trị còn thiếu vào thuộc tính tuổi (age)
Các giá trị thiếu được sinh ngẫu nhiên trong khoảng [trung bình − độ lệch chuẩn, trung bình + độ lệch chuẩn], số lượng bằng số giá trị còn thiếu. Dùng `np.random.seed(42)` để mỗi lần chạy cho cùng kết quả.

In [7]:
np.random.seed(42)

for dataset in [df]:
    mean = df["age"].mean()
    std = df["age"].std()
    is_null = dataset["age"].isnull().sum()
    # compute random numbers between the mean, std and is_null
    rand_age = np.random.randint(mean - std, mean + std, size=is_null)
    # fill NaN values in Age column with random values generated
    age_slice = dataset["age"].copy()
    age_slice[np.isnan(age_slice)] = rand_age
    dataset["age"] = age_slice
    dataset["age"] = df["age"].astype(int)

print('Trung bình:', round(mean, 2), '- Độ lệch chuẩn:', round(std, 2), '- Số giá trị đã điền:', is_null)
print('Số giá trị age còn thiếu:', df["age"].isnull().sum())

Trung bình: 29.88 - Độ lệch chuẩn: 14.41 - Số giá trị đã điền: 263
Số giá trị age còn thiếu: 0


### Bước 7. Điền giá trị còn thiếu vào thuộc tính Cổng lên tàu (embarked)
Xem xét thông tin về thuộc tính này:

In [8]:
df['embarked'].describe()

count     1307
unique       3
top          S
freq       914
Name: embarked, dtype: object

Giá trị `S` là phổ biến nhất (914/1307) nên chọn nó để điền vào các vị trí còn trống.

In [9]:
common_value = 'S'

for dataset in [df]:
    dataset['embarked'] = dataset['embarked'].fillna(common_value)
print('Số giá trị embarked còn thiếu:', df['embarked'].isnull().sum())

Số giá trị embarked còn thiếu: 0


### Bước 8. Điền giá trị còn thiếu vào thuộc tính giá vé (fare)
`fare` chỉ thiếu 1 giá trị → điền bằng trung vị (median) để không bị ảnh hưởng bởi các giá vé ngoại lệ rất lớn.

In [10]:
print('Dòng bị thiếu fare:')
print(df[df['fare'].isnull()][['pclass', 'name', 'embarked', 'fare']])
df['fare'] = df['fare'].fillna(df['fare'].median())
print('\nSố giá trị fare còn thiếu:', df['fare'].isnull().sum())

Dòng bị thiếu fare:
      pclass                name embarked  fare
1225       3  Storey, Mr. Thomas        S   NaN

Số giá trị fare còn thiếu: 0


### Bước 9. Kiểm tra lại dữ liệu sau khi tiền xử lý

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    1309 non-null   int64  
 1   survived  1309 non-null   int64  
 2   name      1309 non-null   str    
 3   sex       1309 non-null   str    
 4   age       1309 non-null   int64  
 5   sibsp     1309 non-null   int64  
 6   parch     1309 non-null   int64  
 7   fare      1309 non-null   float64
 8   embarked  1309 non-null   str    
 9   deck      1309 non-null   int64  
dtypes: float64(1), int64(6), str(3)
memory usage: 102.4 KB


In [12]:
print('Tổng số giá trị còn thiếu:', df.isnull().sum().sum())
df.head()

Tổng số giá trị còn thiếu: 0


,pclass,survived,name,sex,age,sibsp,parch,fare,embarked,deck
0,1,1,"Allen, Miss. Elisabeth Walton",female,29,0,0,211.3375,S,2
1,1,1,"Allison, Master. Hudson Trevor",male,0,1,2,151.5500,S,3
2,1,0,"Allison, Miss. Helen Loraine",female,2,1,2,151.5500,S,3
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30,1,2,151.5500,S,3
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25,1,2,151.5500,S,3


### Nhận xét
- Trước khi xử lý, dữ liệu có **7 thuộc tính bị thiếu**: `body` (1188), `cabin` (1014), `boat` (823), `home.dest` (564), `age` (263), `embarked` (2), `fare` (1).
- **Thuộc tính đã xóa (5):** `cabin` (sau khi tách lấy `deck`), `ticket` (929 giá trị khác nhau, gần như mỗi người một mã), `boat`, `body`, `home.dest` (thiếu quá nhiều dữ liệu, ít liên quan hoặc là thông tin có sau khi biết kết quả sống sót).
- **Thuộc tính mới:** `deck` – boong tàu tách từ `cabin`; phần lớn hành khách không có thông tin cabin nên `deck` = 8 (Unknown) chiếm 1014/1309.
- **Điền giá trị thiếu:**
  - `age`: 263 giá trị được sinh ngẫu nhiên trong khoảng trung bình ± độ lệch chuẩn (29.88 ± 14.41), giúp giữ phân bố tuổi gần với dữ liệu gốc thay vì dồn hết vào một giá trị; sau đó chuyển về kiểu số nguyên.
  - `embarked`: 2 giá trị được điền bằng `S` – giá trị phổ biến nhất (mode).
  - `fare`: 1 giá trị được điền bằng trung vị.
- **Kết quả:** dữ liệu còn **1309 dòng, 10 thuộc tính** và **không còn giá trị thiếu nào**, sẵn sàng cho bước biến đổi thuộc tính ở Câu 7.